<img src="https://raw.githubusercontent.com/OmarLi-360/powerbi-tableau-analista/main/assets/logo-drbits.jpg" alt="Dr. Bits" width="170">

# Clave 2 · Transformar para el modelo
**Analista de datos · Power BI + Tableau** — Hechos, dimensiones y tabla de fechas · ⏱ 25 min

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OmarLi-360/powerbi-tableau-analista/blob/main/notebooks/02_transformar_para_el_modelo.ipynb) &nbsp; [Volver al tablero](https://omarli-360.github.io/powerbi-tableau-analista/#clave-02)

<sub>Si al abrir un enlace Google muestra «Redirect Notice», haz clic en el enlace azul de esa página: es un aviso de seguridad de Google, no un error.</sub>

**Al terminar podrás:**
1. Ver cuántos días no tienen ventas y por qué hace falta un Calendario.
2. Comparar el tamaño de la tabla plana con el del modelo estrella.
3. Entender la diferencia de detalle entre ventas y metas.

---
### Cómo usar este cuaderno (no necesitas programar)
1. **Guarda tu copia:** *Archivo › Guardar una copia en Drive*. Así tus cambios no se pierden. Si no lo haces, no pasa nada grave: el original está protegido y nadie puede dañarlo desde aquí, pero lo que cambies se borra al cerrar.
2. **Ejecuta todo:** *Entorno de ejecución › Ejecutar todas* (Ctrl+F9). Si Colab avisa que el cuaderno no es de Google, elige *Ejecutar de todos modos*.
3. **Juega con los controles:** las celdas con listas desplegables a la derecha se pueden cambiar. Elige una opción y pulsa el botón ▶ de esa celda para redibujar.

El código está oculto a propósito: aquí importa entender el dato y el gráfico. Si tienes curiosidad, haz doble clic en una celda para verlo.

In [ ]:
# @title Preparar el entorno (se ejecuta sola con «Ejecutar todas»)
import os, sys, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, Markdown, HTML
warnings.filterwarnings("ignore")

EN_COLAB = "google.colab" in sys.modules
REPO = "OmarLi-360/powerbi-tableau-analista"
BASE_URL = f"https://raw.githubusercontent.com/{REPO}/main/data/"
BASE_LOCAL = "../data/"

def ruta(nombre):
    local = os.path.join(BASE_LOCAL, nombre)
    return local if os.path.exists(local) else BASE_URL + nombre

# --- formatos colombianos -------------------------------------------------
def miles(x):    return f"{x:,.0f}".replace(",", ".")
def pesos(x):    return "$ " + miles(x)
def millones(x): return "$ " + f"{x/1e6:,.0f}".replace(",", ".") + " M"
def porc(v, signo=False):
    t = f"{v:+.1%}" if signo else f"{v:.1%}"
    return t.replace(".", ",").replace("%", " %")
MESES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
MM = "Millones de pesos"

# --- plantilla de gráficos Dr. Bits ----------------------------------------
AZUL, TEAL, MAGENTA, GRIS = "#4A5AD8", "#0E9BB8", "#D0336F", "#9AA3BF"
BIEN, MAL = "#15784A", "#B42318"
ejes = dict(gridcolor="#E4E8F2", zeroline=False, linecolor="#C9D0E2", exponentformat="none", separatethousands=True)
pio.templates["drbits"] = go.layout.Template(layout=dict(
    font=dict(family="Source Sans 3, Segoe UI, Arial, sans-serif", size=14, color="#11153A"),
    title=dict(font=dict(size=18, color="#00006A"), x=0, xanchor="left"),
    colorway=[AZUL, TEAL, MAGENTA], plot_bgcolor="white", paper_bgcolor="white",
    xaxis=ejes, yaxis=ejes, separators=",.", hoverlabel=dict(font_size=13),
    margin=dict(t=80, l=10, r=30, b=40)))
pio.templates.default = "plotly_white+drbits"

def eje_meses(fig, fechas, cada=None):
    """Etiquetas de meses en español en ejes de tiempo (Plotly las pone en inglés)."""
    f = pd.to_datetime(pd.Series(sorted(set(pd.to_datetime(list(fechas))))))
    paso = cada or (1 if len(f) <= 13 else 3)
    f = f.iloc[::paso]
    fig.update_xaxes(tickvals=list(f), ticktext=[f"{MESES[d.month - 1]} {d.year}" for d in f])

def dar_aire(fig, maximo, eje="x"):
    (fig.update_xaxes if eje == "x" else fig.update_yaxes)(range=[0, maximo * 1.22])

# --- revisor de retos -----------------------------------------------------
def revisar(respuesta, esperado, explicacion=""):
    if respuesta == "Elige una opción":
        display(Markdown("**Elige una opción en la lista de la derecha** y vuelve a ejecutar esta celda con ▶."))
    elif str(respuesta) == str(esperado):
        display(Markdown(f"✅ **Correcto.** {explicacion}"))
    else:
        display(Markdown(f"❌ **Todavía no.** Revisa el gráfico de arriba e inténtalo otra vez. *(Elegiste: {respuesta})*"))

# --- datos de Distribuidora Andina ----------------------------------------
ventas = pd.read_csv(ruta("ventas.csv"), parse_dates=["fecha"])
clientes = pd.read_csv(ruta("clientes.csv"))
productos = pd.read_csv(ruta("productos.csv"))
vendedores = pd.read_csv(ruta("vendedores.csv"))
sedes = pd.read_csv(ruta("sedes.csv"))
metas = pd.read_csv(ruta("metas.csv"))
df = (ventas
      .merge(sedes[["id_sede", "ciudad"]], on="id_sede", validate="many_to_one")
      .merge(productos[["sku", "producto", "categoria"]], on="sku", validate="many_to_one")
      .merge(clientes[["id_cliente", "razon_social", "segmento"]], on="id_cliente", validate="many_to_one")
      .merge(vendedores[["id_vendedor", "nombre_vendedor"]], on="id_vendedor", validate="many_to_one")
      .assign(anio=lambda d: d.fecha.dt.year, mes=lambda d: d.fecha.dt.month))
validas = df[df["estado"] != "Anulada"]          # las anuladas no son venta
ORDEN_CIUDADES = (validas[validas.anio == 2025].groupby("ciudad")["subtotal"].sum()
                  .sort_values(ascending=False).index.tolist())
display(Markdown(f"Datos listos: **{miles(len(ventas))} líneas de venta**, {len(clientes)} clientes, {len(productos)} productos. "
                 f"Fuente: {'archivos locales' if os.path.exists(BASE_LOCAL) else 'GitHub'}."))

## 1 · Los días sin ventas

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Cuántos días de cada mes no tuvieron ventas? |
> | **Qué se grafica** | Barras apiladas por mes: días con ventas y días sin ventas, 2024 y 2025 |
> | **Por qué esta forma** | Muestra que los huecos son regulares (domingos y festivos) y suman más de cien días. |
> | **Qué se descartó** | Solo el total de días sin ventas: no se ve que pasa todos los meses. |
> | **Cómo se lee** | La parte gris son días que no existen en Ventas[fecha]. Por eso el Calendario se crea aparte, con todos los días. |
> | **En Excel 365** | Tabla Calendario con CALENDAR(DATE(2024,1,1), DATE(2026,12,31)) y Marcar como tabla de fechas. |

In [ ]:
# @title Gráfico 1 · Días con y sin ventas
cal = pd.DataFrame({"Date": pd.date_range("2024-01-01", "2025-12-31")})
cal["con_ventas"] = cal.Date.isin(ventas.fecha.dt.normalize())
cal["periodo"] = cal.Date.dt.to_period("M").dt.to_timestamp()
g = cal.groupby("periodo")["con_ventas"].agg(["sum", "count"])
fig = go.Figure()
fig.add_bar(x=g.index, y=g["sum"], name="Con ventas", marker_color=AZUL)
fig.add_bar(x=g.index, y=g["count"] - g["sum"], name="Sin ventas", marker_color=GRIS)
sin = int((~cal.con_ventas).sum())
eje_meses(fig, g.index)
fig.update_layout(barmode="stack", title=f"{sin} de {len(cal)} días de 2024–2025 no tienen ventas: el Calendario los incluye",
                  yaxis_title="Días", height=360, legend=dict(orientation="h", y=-0.2))
fig.show()

## 2 · Tabla plana o modelo estrella

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Cuánto se repite en la tabla plana? |
> | **Qué se grafica** | Barras: celdas de la tabla plana y del modelo estrella, por tabla |
> | **Por qué esta forma** | Compara el tamaño total y muestra que las dimensiones son pequeñas. |
> | **Qué se descartó** | Hablar de «eficiencia» sin cifras. |
> | **Cómo se lee** | En la tabla plana, el segmento y la ciudad se escriben 17.106 veces. En la estrella, una vez por cliente o sede. |
> | **En Excel 365** | Power BI comprime los datos, pero el modelo estrella sigue siendo más claro, más rápido y más fácil de mantener. |

In [ ]:
# @title Gráfico 2 · Celdas en cada diseño
n = len(ventas)
plana = n * 17
estrella = {"Ventas (hechos + llaves)": n * 11, "Clientes": len(clientes) * 3, "Productos": len(productos) * 3,
            "Sedes": len(sedes) * 3, "Vendedores": len(vendedores) * 2, "Calendario": 1096 * 4}
fig = go.Figure()
fig.add_bar(name="Tabla plana", x=["Tabla plana"], y=[plana], marker_color=GRIS, text=[miles(plana)], textposition="outside")
base = 0
for i, (k, v) in enumerate(estrella.items()):
    fig.add_bar(name=k, x=["Modelo estrella"], y=[v], marker_color=[AZUL, TEAL, MAGENTA, "#7A86E8", "#E8A33D", "#5BAA6A"][i])
tot = sum(estrella.values())
fig.update_layout(barmode="stack", title=f"El modelo estrella guarda {porc(1 - tot / plana)} menos celdas ({miles(tot)} frente a {miles(plana)})",
                  yaxis_title="Celdas", height=440, legend=dict(orientation="h", y=-0.15))
dims = tot - estrella["Ventas (hechos + llaves)"]
fig.add_annotation(x="Modelo estrella", y=tot, text=f"{miles(tot)}<br>(dimensiones: {miles(dims)})", showarrow=False, yshift=22)
fig.show()

## 3 · La granularidad: día frente a mes

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Por qué Metas necesita una columna fecha_mes? |
> | **Qué se grafica** | Arriba: venta diaria de 2025 · Abajo: meta mensual de 2025 |
> | **Por qué esta forma** | Dos paneles con el mismo eje de tiempo muestran que una tabla tiene 365 puntos y la otra 12. |
> | **Qué se descartó** | Unir metas y ventas en una tabla: la meta quedaría repetida en cada venta del mes. |
> | **Cómo se lee** | Para comparar, ambas se llevan al nivel del mes: las ventas se suman por mes y la meta se relaciona con el Calendario por el primer día del mes. |
> | **En Excel 365** | Metas[fecha_mes] = DATE(Metas[anio], Metas[mes], 1) y relación con Calendario[Date]. |

In [ ]:
# @title Gráfico 3 · Venta diaria y meta mensual
from plotly.subplots import make_subplots
d = validas[validas.anio == 2025].groupby(validas.fecha.dt.normalize())["subtotal"].sum() / 1e6
m = metas[metas.anio == 2025].groupby("mes")["meta_subtotal"].sum() / 1e6
fm = pd.to_datetime([f"2025-{x:02d}-01" for x in m.index])
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=[f"Ventas: {len(d)} días con datos", "Metas: 12 meses (una fila por sede y mes)"])
fig.add_scatter(x=d.index, y=d.values, mode="lines", line=dict(color=AZUL, width=1), row=1, col=1, showlegend=False)
fig.add_bar(x=fm, y=m.values, marker_color=GRIS, row=2, col=1, showlegend=False)
eje_meses(fig, fm)
fig.update_layout(title="Ventas es diaria y Metas mensual: se comparan al nivel del mes", height=460, yaxis_title=MM)
fig.show()

## 4 · Retos de interpretación

In [ ]:
# @title Reto 1 · ¿Cuántos días de 2024–2025 no tienen ventas?
respuesta = "Elige una opción"  # @param ["Elige una opción", "0", "52", "112", "365"]
revisar(respuesta, str(sin), """Es la parte gris del gráfico 1.""")

In [ ]:
# @title Reto 2 · ¿En qué tabla del modelo va la columna categoría?
respuesta = "Elige una opción"  # @param ["Elige una opción", "Ventas", "Productos", "Calendario", "Sedes"]
revisar(respuesta, "Productos", """Describe al producto: se guarda una vez por producto.""")

In [ ]:
# @title Reto 3 · ¿Qué columna se agrega a Metas para relacionarla con el Calendario?
respuesta = "Elige una opción"  # @param ["Elige una opción", "mes", "fecha_mes = DATE(anio, mes, 1)", "id_linea", "ciudad"]
revisar(respuesta, "fecha_mes = DATE(anio, mes, 1)", """El primer día del mes existe en el Calendario y representa el mes.""")

## Apóyate en IA
Para diseñar el modelo a partir de una tabla plana:

```text
Tengo una tabla plana con estas columnas: [LISTA].
Propón un modelo estrella: tabla de hechos, dimensiones, llaves y la tabla Calendario en DAX.
```

> **Verifica siempre.** Compara la respuesta de la IA con un dato que ya conozcas. No pegues datos personales reales (Ley 1581 de 2012): estos datos son ficticios, los de tu empresa no.

## Resumen
* Hechos con números y llaves; dimensiones con atributos, una vez cada uno.
* El Calendario tiene todos los días, haya o no ventas.
* Tablas con distinto detalle se comparan a través de dimensiones comunes.

**Siguiente:** [Clave 3 · Modelo estrella](https://omarli-360.github.io/powerbi-tableau-analista/#clave-03) · [Cuaderno 03](https://colab.research.google.com/github/OmarLi-360/powerbi-tableau-analista/blob/main/notebooks/03_modelo_estrella.ipynb)

<sub>Dr. Bits · ADNBITS S.A.S. · Contenido CC BY-SA 4.0 · Datos ficticios de Distribuidora Andina S.A.S.</sub>